In [1]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
import os

# 경고 무시 설정
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
# 작업 디렉토리로 이동 (pbp.py가 있는 위치)
%cd C:/Users/user/KBO_pbp_text_crawler

C:\Users\user\KBO_pbp_text_crawler


In [6]:
# 2009년부터 있고, 로케이션 & 무브먼트 추적은 2017년부터 가능
from download import download_pbp_files
import glob
import subprocess

# 크롤링 명령어 구성 (포스트시즌 포함, 통합 저장 포함)
cmd = [
    'python', 'pbp.py',
    '-f', '20180301',     # 시작일
    '-t', '20241231',     # 종료일
    '-p',                 # 포스트시즌 포함
    '--save-source',      # 원본 HTML 저장
    '-g'                  # 디버그 출력
]

# 실행하면서 출력 캡처
result = subprocess.run(cmd, capture_output=True, text=True)

# 표준 출력
print("[STDOUT]")
print(result.stdout)

# 표준 에러
print("[STDERR]")
print(result.stderr)

# 종료 코드
print(f"\n[Return code] {result.returncode}")

[STDOUT]
------------------------------------------------------------
=== gameID : 20180426HHHT02018
=== 8회말 0사
=== text - 정성훈 : 1루수 땅볼 아웃 (1루수 1루 터치아웃)
=== text - 정성훈 : 1루수 땅볼 아웃 (1루수 1루 터치아웃)
------------------------------------------------------------
주자 처리 에러: 루상에서 처리하려는 대상 주자를 찾을 수 없음
------------------------------------------------------------
=== gameID : 20190424LTHH02019
=== 11회말 0사
=== text - 투수 고효준 : 투수 오현택 (으)로 교체
------------------------------------------------------------
교체 에러: 교체하려는 투수가 경기 투수 기록 명단에 없음
------------------------------------------------------------
=== gameID : 20190816SKHT02019
=== 8회말 1사
=== text - 박찬호 : 투수 번트 아웃 (투수->1루수 송구아웃)
=== text - 박찬호 : 투수 번트 아웃 (투수->1루수 송구아웃)
------------------------------------------------------------
주자 처리 에러: 루상에서 처리하려는 대상 주자를 찾을 수 없음
------------------------------------------------------------
=== gameID : 20190929SSKT02019
=== 7회초 0사
=== text - 투수 강백호 : 지명타자 김재윤 (으)로 교체
------------------------------------------------------

In [5]:
import os
import pandas as pd
from glob import glob

# 기준 경로
base_root = r'C:\Users\user\KBO_pbp_text_crawler\save'

# 처리 대상 연도 범위
years = list(range(2017, 2025))

# 연도별 결과 저장
yearly_dfs = {}

# 연도별 반복 처리
for year in years:
    base_dir = os.path.join(base_root, str(year))
    pattern = f'*{year}.csv'
    csv_files = [f for f in glob(os.path.join(base_dir, pattern))]

    csv_list = []

    print(f"\n[INFO] ===== {year}년 처리 시작 =====")
    for file in csv_files:
        try:
            print(f"[INFO] 파일 로드 중: {os.path.basename(file)}")
            try:
                df = pd.read_csv(file, encoding='utf-8')
            except UnicodeDecodeError:
                df = pd.read_csv(file, encoding='cp949')

            df['source_file'] = os.path.basename(file)
            csv_list.append(df)
        except Exception as e:
            print(f"[ERROR] {file}: {e}")

    if csv_list:
        main_df = pd.concat(csv_list, ignore_index=True)
        yearly_dfs[year] = main_df
        print(f"[RESULT] {year}년: 파일 {len(csv_list)}개, shape={main_df.shape}")
    else:
        print(f"[WARNING] {year}년: 병합할 파일 없음")

# 예: 특정 연도 데이터 접근
# df_2020 = yearly_dfs[2020]


[INFO] ===== 2017년 처리 시작 =====
[INFO] 파일 로드 중: 20170331HHOB02017.csv
[INFO] 파일 로드 중: 20170331HTSS02017.csv
[INFO] 파일 로드 중: 20170331KTSK02017.csv
[INFO] 파일 로드 중: 20170331LGWO02017.csv
[INFO] 파일 로드 중: 20170331LTNC02017.csv
[INFO] 파일 로드 중: 20170401HHOB02017.csv
[INFO] 파일 로드 중: 20170401HTSS02017.csv
[INFO] 파일 로드 중: 20170401KTSK02017.csv
[INFO] 파일 로드 중: 20170401LGWO02017.csv
[INFO] 파일 로드 중: 20170401LTNC02017.csv
[INFO] 파일 로드 중: 20170402HHOB02017.csv
[INFO] 파일 로드 중: 20170402HTSS02017.csv
[INFO] 파일 로드 중: 20170402KTSK02017.csv
[INFO] 파일 로드 중: 20170402LGWO02017.csv
[INFO] 파일 로드 중: 20170402LTNC02017.csv
[INFO] 파일 로드 중: 20170404NCHH02017.csv
[INFO] 파일 로드 중: 20170404OBKT02017.csv
[INFO] 파일 로드 중: 20170404SKHT02017.csv
[INFO] 파일 로드 중: 20170404SSLG02017.csv
[INFO] 파일 로드 중: 20170404WOLT02017.csv
[INFO] 파일 로드 중: 20170406NCHH02017.csv
[INFO] 파일 로드 중: 20170406OBKT02017.csv
[INFO] 파일 로드 중: 20170406SKHT02017.csv
[INFO] 파일 로드 중: 20170406SSLG02017.csv
[INFO] 파일 로드 중: 20170406WOLT02017.csv
[INFO] 파일 로드 중: 20

In [9]:
# 저장 경로 (원하는 경로로 수정 가능)
output_dir = r'C:\Users\user\KBO_pbp_text_crawler\merged_output'
os.makedirs(output_dir, exist_ok=True)

# 연도별 DataFrame 저장
for year, df in yearly_dfs.items():
    output_path = os.path.join(output_dir, f'merged_{year}.csv')
    try:
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"[SAVED] {output_path} (rows: {df.shape[0]})")
    except Exception as e:
        print(f"[ERROR] 저장 실패 - {year}: {e}")

[SAVED] C:\Users\user\KBO_pbp_text_crawler\merged_output\merged_2017.csv (rows: 234593)
[SAVED] C:\Users\user\KBO_pbp_text_crawler\merged_output\merged_2018.csv (rows: 236728)
[SAVED] C:\Users\user\KBO_pbp_text_crawler\merged_output\merged_2019.csv (rows: 225952)
[SAVED] C:\Users\user\KBO_pbp_text_crawler\merged_output\merged_2020.csv (rows: 236568)
[SAVED] C:\Users\user\KBO_pbp_text_crawler\merged_output\merged_2021.csv (rows: 236114)
[SAVED] C:\Users\user\KBO_pbp_text_crawler\merged_output\merged_2022.csv (rows: 232871)
[SAVED] C:\Users\user\KBO_pbp_text_crawler\merged_output\merged_2023.csv (rows: 235957)
[SAVED] C:\Users\user\KBO_pbp_text_crawler\merged_output\merged_2024.csv (rows: 239729)
